### 2014 ~ 2019

# 2019

In [1]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [6]:
import re
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 ICDM 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기 (electronic edition via DOI)
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        if doi_tag and doi_tag.get("href"):
            pdf_link = doi_tag["href"]
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [10]:
url = 'https://dblp.org/db/conf/cvpr/cvpr2019.html'
DB_PATH = "con_db/CVPR_conference_2019.db"
conference_name = 'CVPR 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [11]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/CVPR_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [12]:
df_papers = get_www_papers('html/CVPR_2019_accepted_papers.html', conference_name)

In [13]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Finding Task-Relevant Features for Few-Shot Le...,"Hongyang Li, David Eigen, Samuel Dodge, Matthe...",https://doi.org/10.1109/CVPR.2019.00009,None,CVPR 2019
1,Edge-Labeling Graph Neural Network for Few-Sho...,"Jongmin Kim, Taesup Kim, Sungwoong Kim, Chang ...",https://doi.org/10.1109/CVPR.2019.00010,None,CVPR 2019
2,Generating Classification Weights With GNN Den...,"Spyros Gidaris, Nikos Komodakis",https://doi.org/10.1109/CVPR.2019.00011,None,CVPR 2019
3,Kervolutional Neural Networks.,"Chen Wang, Jianfei Yang, Lihua Xie, Junsong Yuan",https://doi.org/10.1109/CVPR.2019.00012,None,CVPR 2019
4,Why ReLU Networks Yield High-Confidence Predic...,"Matthias Hein, Maksym Andriushchenko, Julian B...",https://doi.org/10.1109/CVPR.2019.00013,None,CVPR 2019


In [14]:
save_to_database(df_papers, conference_name, DB_PATH)

1294개의 논문이 CVPR 2019에 저장되었습니다.


# 2018

In [15]:
url = 'https://dblp.org/db/conf/cvpr/cvpr2018.html'
DB_PATH = "con_db/CVPR_conference_2018.db"
conference_name = 'CVPR 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [16]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/CVPR_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [17]:
df_papers = get_www_papers('html/CVPR_2018_accepted_papers.html', conference_name)

In [18]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Embodied Question Answering.,"Abhishek Das, Samyak Datta, Georgia Gkioxari, ...",https://doi.org/10.1109/CVPR.2018.00008,None,CVPR 2018
1,Learning by Asking Questions.,"Ishan Misra, Ross B. Girshick, Rob Fergus, Mar...",https://doi.org/10.1109/CVPR.2018.00009,None,CVPR 2018
2,Finding Tiny Faces in the Wild With Generative...,"Yancheng Bai, Yongqiang Zhang, Mingli Ding, Be...",https://doi.org/10.1109/CVPR.2018.00010,None,CVPR 2018
3,Learning Face Age Progression: A Pyramid Archi...,"Hongyu Yang, Di Huang, Yunhong Wang, Anil K. Jain",https://doi.org/10.1109/CVPR.2018.00011,None,CVPR 2018
4,PairedCycleGAN: Asymmetric Style Transfer for ...,"Huiwen Chang, Jingwan Lu, Fisher Yu, Adam Fink...",https://doi.org/10.1109/CVPR.2018.00012,None,CVPR 2018


In [19]:
save_to_database(df_papers, conference_name, DB_PATH)

979개의 논문이 CVPR 2018에 저장되었습니다.


# 2017

In [21]:
url = 'https://dblp.org/db/conf/cvpr/cvpr2017.html'
DB_PATH = "con_db/CVPR_conference_2017.db"
conference_name = 'CVPR 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [22]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/CVPR_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [23]:
df_papers = get_www_papers('html/CVPR_2017_accepted_papers.html', conference_name)

In [24]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Exclusivity-Consistency Regularized Multi-view...,"Xiaobo Wang, Xiaojie Guo, Zhen Lei, Changqing ...",https://doi.org/10.1109/CVPR.2017.8,None,CVPR 2017
1,Borrowing Treasures from the Wealthy: Deep Tra...,"Weifeng Ge, Yizhou Yu",https://doi.org/10.1109/CVPR.2017.9,None,CVPR 2017
2,The More You Know: Using Knowledge Graphs for ...,"Kenneth Marino, Ruslan Salakhutdinov, Abhinav ...",https://doi.org/10.1109/CVPR.2017.10,None,CVPR 2017
3,Dynamic Edge-Conditioned Filters in Convolutio...,"Martin Simonovsky, Nikos Komodakis",https://doi.org/10.1109/CVPR.2017.11,None,CVPR 2017
4,Convolutional Neural Network Architecture for ...,"Ignacio Rocco, Relja Arandjelovic, Josef Sivic",https://doi.org/10.1109/CVPR.2017.12,None,CVPR 2017


In [25]:
save_to_database(df_papers, conference_name, DB_PATH)

783개의 논문이 CVPR 2017에 저장되었습니다.


# 2016

In [27]:
url = 'https://dblp.org/db/conf/cvpr/cvpr2016.html'
DB_PATH = "con_db/CVPR_conference_2016.db"
conference_name = 'CVPR 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [28]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/CVPR_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [29]:
df_papers = get_www_papers('html/CVPR_2016_accepted_papers.html', conference_name)

In [30]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Deep Compositional Captioning: Describing Nove...,"Lisa Anne Hendricks, Subhashini Venugopalan, M...",https://doi.org/10.1109/CVPR.2016.8,None,CVPR 2016
1,Generation and Comprehension of Unambiguous Ob...,"Junhua Mao, Jonathan Huang, Alexander Toshev, ...",https://doi.org/10.1109/CVPR.2016.9,None,CVPR 2016
2,Stacked Attention Networks for Image Question ...,"Zichao Yang, Xiaodong He, Jianfeng Gao, Li Den...",https://doi.org/10.1109/CVPR.2016.10,None,CVPR 2016
3,Image Question Answering Using Convolutional N...,"Hyeonwoo Noh, Paul Hongsuck Seo, Bohyung Han",https://doi.org/10.1109/CVPR.2016.11,None,CVPR 2016
4,Neural Module Networks.,"Jacob Andreas, Marcus Rohrbach, Trevor Darrell...",https://doi.org/10.1109/CVPR.2016.12,None,CVPR 2016


In [31]:
save_to_database(df_papers, conference_name, DB_PATH)

643개의 논문이 CVPR 2016에 저장되었습니다.


# 2015

In [32]:
url = 'https://dblp.org/db/conf/cvpr/cvpr2015.html'
DB_PATH = "con_db/CVPR_conference_2015.db"
conference_name = 'CVPR 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [33]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/CVPR_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [34]:
df_papers = get_www_papers('html/CVPR_2015_accepted_papers.html', conference_name)

In [35]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Going deeper with convolutions.,"Christian Szegedy, Wei Liu, Yangqing Jia, Pier...",https://doi.org/10.1109/CVPR.2015.7298594,None,CVPR 2015
1,Propagated image filtering.,"Jen-Hao Rick Chang, Yu-Chiang Frank Wang",https://doi.org/10.1109/CVPR.2015.7298595,None,CVPR 2015
2,Web scale photo hash clustering on a single ma...,"Yunchao Gong, Marcin Pawlowski, Fei Yang, Loui...",https://doi.org/10.1109/CVPR.2015.7298596,None,CVPR 2015
3,Expanding object detector's Horizon: Increment...,"Alina Kuznetsova, Sung Ju Hwang, Bodo Rosenhah...",https://doi.org/10.1109/CVPR.2015.7298597,None,CVPR 2015
4,Supervised Discrete Hashing.,"Fumin Shen, Chunhua Shen, Wei Liu, Heng Tao Shen",https://doi.org/10.1109/CVPR.2015.7298598,None,CVPR 2015


In [36]:
save_to_database(df_papers, conference_name, DB_PATH)

601개의 논문이 CVPR 2015에 저장되었습니다.
